## Reestructuración Generativa (K=11)
### Objetivos y Metodología del Experimento

En esta fase, utilizamos el algoritmo **K-Means** sobre los embeddings anonimizados para agrupar los párrafos en 11 categorías. El objetivo es evaluar la coherencia de estos grupos mediante un **Comité de Modelos de Lenguaje (LLMs)** locales ejecutados en Ollama:
* **Llama3:latest**
* **Llama3_hard** (versión con parámetros de temperatura ajustados)
* **Qwen2:8b**

**Procedimiento:** Para cada cluster, se selecciona una muestra representativa de párrafos y se le solicita a los modelos que generen una **Pregunta Maestra** universal que resuma la intención comunicativa del grupo.

#### Extracción de prospectos

In [ ]:
import requests
import time
import pandas as pd

def extraer_datos_cima_unicos(limit=200):
    base_url = "https://cima.aemps.es/cima/rest"
    headers = {"Accept": "application/json"}
    prospectos_lista = []
    medicamentos_vistos = set() # Aquí guardamos la "primera palabra" del nombre del medicamento
    
    pagina = 1
    total_capturados = 0

    while total_capturados < limit:
        params_busqueda = {'pagina': pagina, 'prospecto': 1}
        r_listado = requests.get(f"{base_url}/medicamentos", params=params_busqueda)
        
        if r_listado.status_code != 200: break
        medicamentos = r_listado.json().get('resultados', [])
        
        if not medicamentos: break # No hay más resultados

        for med in medicamentos:
            if total_capturados >= limit: break
            
            # 1. Normalizamos el nombre para el check
            # Tomamos la primera palabra
            nombre_raiz = med['nombre'].split()[0].upper()
            
            # 2. Check de unicidad
            if nombre_raiz in medicamentos_vistos:
                continue # Saltamos al siguiente medicamento de la lista
            
            nreg = med['nregistro']
            url_doc = f"{base_url}/docSegmentado/contenido/2"
            
            try:
                r_doc = requests.get(url_doc, params={'nregistro': nreg}, headers=headers)
                
                if r_doc.status_code == 200:
                    secciones = r_doc.json() 
                    if not secciones: continue
                    
                    for sec in secciones:
                        prospectos_lista.append({
                            'nregistro': nreg,
                            'nombre_med': med['nombre'],
                            'nombre_raiz': nombre_raiz,
                            'id_seccion_original': sec.get('seccion'),
                            'titulo_original': sec.get('titulo'),
                            'texto': sec.get('contenido')
                        })
                    
                    medicamentos_vistos.add(nombre_raiz)
                    total_capturados += 1
                    print(f"[{total_capturados}/{limit}] Nuevo: {nombre_raiz} ({med['nombre']})")
                
            except Exception as e:
                print(f"Error en {nreg}: {e}")
            
            time.sleep(0.1)
        
        pagina += 1
    
    return pd.DataFrame(prospectos_lista)

df_prospectos = extraer_datos_cima_unicos(500)

[1/500] Nuevo: A.A.S. (A.A.S. 100 mg COMPRIMIDOS)
[2/500] Nuevo: ABACAVIR/LAMIVUDINA (ABACAVIR/LAMIVUDINA DR. REDDYS 600 MG/300 MG COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[3/500] Nuevo: ABIRATERONA (ABIRATERONA GLENMARK 500 MG COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[4/500] Nuevo: ABRAXANE (ABRAXANE 5 MG/ML POLVO PARA DISPERSION PARA PERFUSION)
[5/500] Nuevo: ABRILIA (ABRILIA 20 MG COMPRIMIDOS EFG)
[6/500] Nuevo: ABRYSVO (ABRYSVO POLVO Y DISOLVENTE PARA SOLUCION INYECTABLE)
[7/500] Nuevo: ABSORCOL (ABSORCOL 10 mg COMPRIMIDOS)
[8/500] Nuevo: ACABEL (ACABEL RAPID 8 mg COMPRIMIDOS RECUBIERTOS CON PELICULA)
[9/500] Nuevo: ACALKA (ACALKA 1080 mg COMPRIMIDOS DE LIBERACION PROLONGADA)
[10/500] Nuevo: ACARBOSA (ACARBOSA TECNIGEN 100 mg COMPRIMIDOS)
[11/500] Nuevo: ACECLOFENACO (ACECLOFENACO CINFA 100 mg COMPRIMIDOS RECUBIERTOS CON PELICULA EFG)
[12/500] Nuevo: ACEOTO (ACEOTO PLUS 3 mg/ml + 0,25 mg/ml GOTAS ÓTICAS EN SOLUCIÓN)
[13/500] Nuevo: ACETATO (ACETATO DE CIPROTERONA/ETINILESTRADIOL 

In [19]:
df_prospectos

,nregistro,nombre_med,nombre_raiz,id_seccion_original,titulo_original,texto
0,42991,A.A.S. 100 mg COMPRIMIDOS,A.A.S.,0,Introducción,"<div>\r\n <p style=""margin:10pt 0pt 0pt; te..."
1,42991,A.A.S. 100 mg COMPRIMIDOS,A.A.S.,1,Qué es A.A.S. 100 mg y para qué se utiliza,"<div>\r\n <p style=""margin:0pt; widows:0; o..."
2,42991,A.A.S. 100 mg COMPRIMIDOS,A.A.S.,2,Qué necesita saber antes de empezar a tomar A....,"<div>\r\n <p style=""margin:0pt; widows:0; o..."
3,42991,A.A.S. 100 mg COMPRIMIDOS,A.A.S.,3,Cómo tomar A.A.S. 100 mg,"<div>\r\n <p style=""margin:0pt; text-align:..."
4,42991,A.A.S. 100 mg COMPRIMIDOS,A.A.S.,4,Posibles efectos adversos,"<div>\r\n <p style=""margin:0pt; text-align:..."
...,...,...,...,...,...,...
3495,84048,DUPLAXIL 400 MG COMPRIMIDOS RECUBIERTOS CON PE...,DUPLAXIL,2,Qué necesita saber antes de empezar a tomar Du...,"<div>\r\n <p style=""margin:0pt; text-align:..."
3496,84048,DUPLAXIL 400 MG COMPRIMIDOS RECUBIERTOS CON PE...,DUPLAXIL,3,Cómo tomar Duplaxil,"<div>\r\n <p style=""margin:0pt; text-align:..."
3497,84048,DUPLAXIL 400 MG COMPRIMIDOS RECUBIERTOS CON PE...,DUPLAXIL,4,Posibles efectos adversos,"<div>\r\n <p style=""margin:0pt; text-align:..."
3498,84048,DUPLAXIL 400 MG COMPRIMIDOS RECUBIERTOS CON PE...,DUPLAXIL,5,Conservación de Duplaxil,"<div>\r\n <p style=""margin:0pt; text-align:..."


In [ ]:
from bs4 import BeautifulSoup
import re
import pandas as pd

from bs4 import BeautifulSoup
import re
import pandas as pd

def segmentar_con_contexto(df):
    filas_parrafos = []
    
    for _, row in df.iterrows():
        html = row['texto']
        if not html or pd.isna(html): continue
        
        soup = BeautifulSoup(html, "html.parser")
        
        for script in soup(["script", "style"]):
            script.decompose()

        # Buscamos elementos que suelen ser contenedores de ideas completas
        elementos = soup.find_all(['p', 'ul', 'ol', 'h3', 'h4'])
        
        i = 0
        while i < len(elementos):
            el = elementos[i]
            texto_bloque = el.get_text(separator=' ', strip=True)
  
            # Si un párrafo termina en ":" es un encabezado. 
            # Lo pegamos con el siguiente elemento (que suele ser una lista <ul>)
            if (texto_bloque.endswith(':') or "si se encuentra" in texto_bloque.lower()) and (i + 1 < len(elementos)):
                siguiente_el = elementos[i+1]
                texto_siguiente = siguiente_el.get_text(separator=' ', strip=True)
                
                # Unimos encabezado + lista
                texto_final = f"{texto_bloque} {texto_siguiente}"
                i += 2 
            else:
                texto_final = texto_bloque
                i += 1
            
            # Limpieza final del bloque unido
            texto_final = re.sub(r'\s+', ' ', texto_final).strip()
            
            # Filtro de calidad (evitar frases muy cortas)
            if len(texto_final) > 50:
                filas_parrafos.append({
                    'nregistro': row['nregistro'],
                    'nombre_med': row['nombre_med'],
                    'titulo_original': row['titulo_original'],
                    'parrafo_limpio': texto_final
                })
                
    return pd.DataFrame(filas_parrafos)


df_parrafos = segmentar_con_contexto(df_prospectos)


df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_limpio'])

#### Anonimización de medicamentos

Para evitar sobreajustes sustituimos los nombres de medicamentos por una máscara $[MEDICAMENTO]$.

In [ ]:
import re

def preparar_lista_limpieza(df):
    nombres_sucios = df['nombre_med'].unique().tolist()
    lista_final = set()

    for nombre in nombres_sucios:
        # Añadimos el nombre completo original
        lista_final.add(nombre.strip())
        
        # Extraemos la marca (normalmente la primera palabra)
        marca = nombre.split()[0]
        if len(marca) > 3: # Evitamos palabras muy cortas
            lista_final.add(marca)
            
        # Extraemos todo lo que haya antes de la primera cifra (dosis)
        antes_dosis = re.split(r'\d', nombre)[0].strip()
        if len(antes_dosis) > 3:
            lista_final.add(antes_dosis)

    # Ordenamos de más largo a más corto para que el regex no rompa palabras
    return sorted(list(lista_final), key=len, reverse=True)

def anonimizar_texto(texto, lista_limpieza):
    if not texto: return ""
    
    # Usamos una sola pasada de regex para todas las marcas 
    # Escapamos los nombres por si tienen puntos o paréntesis como "A.A.S."
    patron = r'\b(' + '|'.join(map(re.escape, lista_limpieza)) + r')\b'
    
    # Reemplazamos ignorando mayúsculas/minúsculas
    return re.sub(patron, "[MEDICAMENTO]", texto, flags=re.IGNORECASE)


lista_negra = preparar_lista_limpieza(df_parrafos)
lista_negra += ["A.A.S. 100 mg"]

# Aplicamos la limpieza a la columna de párrafos
df_parrafos['parrafo_anonimizado'] = df_parrafos['parrafo_limpio'].apply(
    lambda x: anonimizar_texto(x, lista_negra)
)


In [24]:
df_parrafos[:10]

,nregistro,nombre_med,titulo_original,parrafo_limpio,parrafo_anonimizado
0,42991,A.A.S. 100 mg COMPRIMIDOS,Introducción,Lea todo el prospecto detenidamente antes de e...,Lea todo el prospecto detenidamente antes de e...
1,42991,A.A.S. 100 mg COMPRIMIDOS,Introducción,"Conserve este prospecto, ya que puede tener qu...","Conserve este prospecto, ya que puede tener qu..."
2,42991,A.A.S. 100 mg COMPRIMIDOS,Introducción,2. Qué necesita saber antes de empezar a tomar...,2. Qué necesita saber antes de empezar a tomar...
3,42991,A.A.S. 100 mg COMPRIMIDOS,Qué es A.A.S. 100 mg y para qué se utiliza,A.A.S. 100 mg está indicado para prevenir la f...,[MEDICAMENTO] está indicado para prevenir la f...
4,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,No tome A.A.S. 100 mg: Si es alérgico al ácido...,No tome [MEDICAMENTO]: Si es alérgico al ácido...
5,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,Consulte a su médico o farmacéutico antes de e...,Consulte a su médico o farmacéutico antes de e...
6,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,Padece alguna enfermedad del riñón (en caso de...,Padece alguna enfermedad del riñón (en caso de...
7,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,"El síndrome de Reye, una enfermedad muy rara p...","El síndrome de Reye, una enfermedad muy rara p..."
8,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,"Consulte a su médico, incluso si cualquiera de...","Consulte a su médico, incluso si cualquiera de..."
9,42991,A.A.S. 100 mg COMPRIMIDOS,Qué necesita saber antes de empezar a tomar A....,Informe a su médico o farmacéutico si está uti...,Informe a su médico o farmacéutico si está uti...


Guardamos el archivo para futuras pruebas

In [25]:
df_parrafos.to_csv("prospectos_limpio_segmentado.csv", index=False)

Volvemos a aplicar una eliminación de duplicados ahora que tenemos los nombres de medicamentos anonimizados.

In [26]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_limpio_segmentado.csv")

df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

### Ejecución del Agrupamiento (k=11)
Seleccionamos $k=11$ como punto de partida basándonos en los resultados previos de los estadísticos de validación (Silhouette y Elbow Method). En esta configuración, buscamos secciones robustas que aparezcan de forma recurrente en todos los modelos.


In [27]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 11 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 261.65it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 969/969 [13:48<00:00,  1.17it/s]


In [28]:
df_parrafos.to_csv("prospectos_clusters_segmentado.csv", index=False)

In [29]:
df_parrafos[df_parrafos["cluster"]==0]["parrafo_anonimizado"]

3        [MEDICAMENTO] está indicado para prevenir la f...
11       Presión arterial alta (diuréticos, antagonista...
17       Si consume habitualmente alcohol (tres o más b...
37       Puede aparecer edema pulmonar no cardiogénico ...
44       Trastornos de la sangre: Alteración de la coag...
                               ...                        
32433    Artritis reumatoide, lupus eritematoso: Debido...
32458    Puede ser necesario ajustar la dosis durante e...
32482    - Inhibición de la formación de vasos sanguíne...
32483    - Cambios en la sangre, tales como un número r...
32489    - Cambios en el ritmo cardíaco (su médico pued...
Name: parrafo_anonimizado, Length: 2075, dtype: str

## Conexión con LLMs

Creamos un prompt que devuelva una pregunta universal para cada cluster de frases. 

In [30]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      
      # PROMPT REFINADO: Foco absoluto en el usuario final
  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿El paracetamol afecta a la capacidad de conducción?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me lo tengo que tomar?".
  3. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Genera la PREGUNTA DIRECTA que un paciente normal (no experto) haría. 
  Ejemplos de estilo correcto: 
  - "¿Qué hago si se me olvida una dosis?" 
  - "¿Puedo conducir después de tomar esto?"
  - "¿Dónde debo guardar la medicina?"

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

#### Conexión con Ollama

Nos conectamos al servidor de Ollama para poder hacer uso de los modelos de lenguaje. 

In [31]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=150) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

In [32]:
import requests

def listar_modelos_disponibles():
    url_base = "https://wiig.dia.fi.upm.es/ollama/api/tags"
    try:
        response = requests.get(url_base, timeout=10)
        if response.status_code == 200:
            modelos = [m['name'] for m in response.json().get('models', [])]
            print(f" Modelos disponibles: {modelos}")
            return modelos
        else:
            print(f" No se pudo listar modelos. Status: {response.status_code}")
    except Exception as e:
        print(f" Error de conexión: {e}")
    return []

modelos_reales = listar_modelos_disponibles()

 Modelos disponibles: ['qwen3:1.7b', 'qwen3:8b', 'llama3_medium:latest', 'llama3_hard:latest', 'llama3_hard_Open:latest', 'llama3_medium_Open:latest', 'llama3_easy:latest', 'llama3_easy_Open:latest', 'llama3:latest']


In [33]:
anotadores = ["llama3:latest", "llama3_medium:latest", "llama3_hard:latest"]
resultados = []

#### Ejecución de llamadas a los LLMs

Una vez elegidos tres modelos hacemos llamadas iterativamente para cada cluster. Así, conseguimos una pregunta por modelo y cluster. 

In [ ]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1)

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_11_topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando llama3:latest...
{
  "pregunta_del_paciente": "¿Qué hago si tengo alguna enfermedad cronica como problemas renales o hepáticos?"
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Cómo puedo reducir el riesgo de coágulos sanguíneos y otros efectos secundarios al tomar este medicamento?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Hay algún riesgo para la salud si tengo antecedentes de ataque al corazón y tomo este medicamento?"
}
 Procesando Cluster 1...
  -> Consultando llama3:latest...
Here is the direct question a patient would ask:

{
"pregunta_del_paciente": "¿Qué sucede si tomo medicamentos que no están relacionados con hidroxicloroquina y ellos interactúan entre sí?"
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Qué hago si me olvido tomar una dosis y no tengo idea de cómo afecta esto con otros medicamentos que estoy tomando?"
}
  -> Consultando llama3_hard:latest..

Extraemos y limpiamos las preguntas devueltas.

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"

  
    # Busca el JSON
    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas 
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)
            
            # Buscamos la pregunta en las posibles llaves que el LLM haya inventado
            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # Si falla el JSON, buscamos patrones de texto
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        # Devolvemos la última pregunta encontrada
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"



df_resultados = pd.read_csv('resultados_etiquetado_11_topicos.csv')

# Columnas de los anotadores
anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    # Creamos una columna limpia para cada modelo
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)


columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())

df_resultados[columnas_formulario].to_csv('preguntas_para_formulario.csv', index=False)

   cluster                             pregunta_llama3_latest  \
0        0  ¿Qué hago si tengo alguna enfermedad cronica c...   
1        1  ¿Qué sucede si tomo medicamentos que no están ...   
2        2      ¿debo consultar con mi médico o farmacéutico?   
3        3  ¿Qué hago si me da un error al tomar [MEDICAME...   
4        4                  ¿Qué hago si me olvido una dosis?   

                       pregunta_llama3_medium_latest  \
0  ¿Cómo puedo reducir el riesgo de coágulos sang...   
1  ¿Qué hago si me olvido tomar una dosis y no te...   
2  ¿Hay algún caso en el que no deba usar [MEDICA...   
3  ¿Qué hago si experimento una deficiencia de gl...   
4   ¿Qué hago si me olvido una dosis de medicamento?   

                         pregunta_llama3_hard_latest  
0  ¿Hay algún riesgo para la salud si tengo antec...  
1                 No se pudo identificar la pregunta  
2  ¿Cómo hago si me doy cuenta que estoy desarrol...  
3  ¿Qué efectos secundarios pueden ocurrir en beb...

### Evaluación Crítica de los 11 Clusters
Tras analizar las preguntas generadas por el comité de LLMs, se extraen las siguientes conclusiones sobre la estabilidad del modelo:

####  Secciones Robustas (Consenso Total)
Se han identificado "secciones puras" donde los tres modelos coinciden casi palabra por palabra:
* **Cluster 4 (Olvido de dosis):** "¿Qué hago si se me olvida una dosis?".
* **Cluster 7 (Emergencias/Alergias):** Acción inmediata ante reacciones graves.
* **Cluster 9 (Efectos secundarios):** Diferenciación entre efectos normales y peligrosos.

####  Inestabilidades y Limitaciones Detectadas
A pesar de los éxitos locales, el modelo de 11 clusters presenta problemas estructurales que impiden su uso definitivo:
1. **Redundancia Estructural:** El tema del "Olvido de dosis" aparece disperso en los clusters 4, 6, 8 y 10, lo que indica que el algoritmo separa por contexto (forma farmacéutica) pero no por intención del paciente.
2. **Presencia de Ruido (Paja):** El Cluster 5 agrupa párrafos excesivamente específicos (geles, colirios) que no logran una abstracción útil.
3. **Temas Diluidos:** Variables críticas de seguridad como la **Conducción** (Cluster 10) y el **Embarazo** (Cluster 6) aparecen mezcladas con otros temas o rodeadas de ruido, perdiendo claridad para el usuario.

**Conclusión:** El modelo de 11 clusters es **inestable**. La mezcla de temas críticos con información secundaria justifica la exploración de un mayor número de clústeres ($k$) para lograr una "fisión informativa" que aísle correctamente la seguridad del paciente.